In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import re
import time
import pandas as pd
from openpyxl import load_workbook

In [62]:
# Initialize the Chrome WebDriver with ChromeDriverManager
driver = webdriver.Chrome(service=webdriver.chrome.service.Service(ChromeDriverManager().install()))
driver

<selenium.webdriver.chrome.webdriver.WebDriver (session="a141cb695926d885b280809d4a2ec983")>

In [64]:
url = 'https://calendar.google.com/calendar/embed?src=o15nqks84cotcpjugj15c5ioj4%40group.calendar.google.com&ctz=America%2FLima'
driver.get( url )

In [55]:
def extraer_eventos_con_datos(driver):
    try:
        # Obtener el mes y año actual de la página
        try:
            mes_y_ano = driver.find_element(By.CLASS_NAME, "UyW9db").text.strip()
        except Exception:
            mes_y_ano = "Sin información"  # Si no se encuentra, asignar valor por defecto
        
        # Encontrar todos los bloques de eventos (divs con class "qLWd9c")
        bloques_eventos = driver.find_elements(By.CLASS_NAME, "qLWd9c")
        
        datos = []
        
        # Iterar por cada bloque de evento
        for bloque in bloques_eventos:
            try:
                # 1. Extraer el número de eventos, día de la semana y fecha
                info_dia = bloque.find_element(By.CLASS_NAME, "CqwSk.XuJrye").text.strip()
                
                # 2. Dentro del bloque, encontrar los eventos individuales
                eventos = bloque.find_elements(By.CLASS_NAME, "nHqeVd")
                
                for evento in eventos:
                    # a. Intentar extraer la hora (si no existe, usar "N.A.")
                    try:
                        hora = evento.find_element(By.CLASS_NAME, "DvyQhe.BdCDHc").text.strip()
                    except Exception:
                        hora = "N.A."  # Rellenar con "N.A." si no hay hora
                    
                    # b. Extraer la descripción y lugar
                    try:
                        descripcion_lugar = evento.find_element(By.CLASS_NAME, "WBi6vc").text.strip()
                    except Exception:
                        descripcion_lugar = "Sin información"  # Rellenar si no hay descripción
                    
                    # Almacenar los datos en un diccionario
                    datos.append({
                        "num_eventos,dia_semana,fecha": info_dia,
                        "hora": hora,
                        "descripción": descripcion_lugar,
                        "mes_ano,ano": mes_y_ano
                        
                    })
            except Exception as e:
                print(f"Error procesando un bloque de eventos: {e}")
        
        print(f"Se extrajeron {len(datos)} eventos.")
        return datos
    
    except Exception as e:
        print(f"Error al extraer eventos: {e}")
        return []

def guardar_eventos_en_csv(datos, nombre_archivo="eventos.csv"):
    if datos:
        df = pd.DataFrame(datos)
        df.to_csv(nombre_archivo, index=False)
        print(f"Datos guardados en {nombre_archivo}.")
    else:
        print("No hay datos para guardar.")

# Llamada de ejemplo (suponiendo que `driver` ya esté definido y cargado en la página adecuada)
eventos_extraidos = extraer_eventos_con_datos(driver)
guardar_eventos_en_csv(eventos_extraidos)

Se extrajeron 57 eventos.
Datos guardados en eventos.csv.


#Este codigo funciona, falta añadir la sepación del mes

In [33]:
def extraer_eventos_con_datos(driver):
    try:
        # Obtener el mes y año actual de la página
        try:
            mes_y_ano = driver.find_element(By.CLASS_NAME, "UyW9db").text.strip()
            anio = mes_y_ano[-4:] if mes_y_ano[-4:].isdigit() else "N.A."
        except Exception:
            mes_y_ano = "Sin información"  # Si no se encuentra, asignar valor por defecto
            anio = "N.A."
        
        # Encontrar todos los bloques de eventos (divs con class "qLWd9c")
        bloques_eventos = driver.find_elements(By.CLASS_NAME, "qLWd9c")
        
        datos = []
        
        # Iterar por cada bloque de evento
        for bloque in bloques_eventos:
            try:
                # 1. Extraer el número de eventos, día de la semana y fecha
                info_dia = bloque.find_element(By.CLASS_NAME, "CqwSk.XuJrye").text.strip()
                
                # 2. Dentro del bloque, encontrar los eventos individuales
                eventos = bloque.find_elements(By.CLASS_NAME, "nHqeVd")
                
                for evento in eventos:
                    # a. Intentar extraer la hora (si no existe, usar "N.A.")
                    try:
                        hora = evento.find_element(By.CLASS_NAME, "DvyQhe.BdCDHc").text.strip()
                    except Exception:
                        hora = "N.A."  # Rellenar con "N.A." si no hay hora
                    
                    # b. Extraer la descripción y lugar
                    try:
                        descripcion_lugar = evento.find_element(By.CLASS_NAME, "WBi6vc")
                    except Exception:
                        descripcion_lugar = "Sin información"  # Rellenar si no hay descripción
                    
                    # Almacenar los datos en un diccionario
                    datos.append({
                        "num_eventos,dia_semana,fecha": info_dia,
                        "hora": hora,
                        "descripción": descripcion_lugar,
                        "anio": anio,               # Año extraído del mes_y_ano
                        "mes_ano,ano": mes_y_ano
                    })
            except Exception as e:
                print(f"Error procesando un bloque de eventos: {e}")
        
        print(f"Se extrajeron {len(datos)} eventos.")
        return datos
    
    except Exception as e:
        print(f"Error al extraer eventos: {e}")
        return []

def guardar_datos_incrementales(datos, archivo_csv="eventos_incrementales.csv"):
    if datos:
        df = pd.DataFrame(datos)
        with open(archivo_csv, 'a', encoding='utf-8', newline='') as f:
            # Agregar encabezado solo si el archivo está vacío
            df.to_csv(f, index=False, header=f.tell() == 0)
        print(f"Datos guardados incrementalmente en {archivo_csv}.")
    else:
        print("No hay datos para guardar.")

def retroceder_mes(driver):
    try:
        boton_anterior = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//button[@aria-label="Mes anterior"]'))
        )
        boton_anterior.click()
        print("Retrocedimos al mes anterior.")
        # Esperar para permitir que la página cargue
        time.sleep(10)
    except Exception as e:
        print(f"Error al retroceder al mes anterior: {e}")

def iterar_y_guardar(driver, fecha_limite_texto, archivo_csv="eventos_incrementales.csv"):
    while True:
        try:
            # Obtener el mes y año actual
            try:
                mes_y_ano = driver.find_element(By.CLASS_NAME, "UyW9db").text.strip()
            except Exception:
                print("No se pudo obtener mes y año, terminando iteración.")
                break
            
            # Verificar si llegamos a la fecha límite
            if mes_y_ano <= fecha_limite_texto:
                print(f"Fecha límite alcanzada ({fecha_limite_texto}), terminando iteración.")
                break
            
            # Extraer eventos del mes actual
            eventos_mes = extraer_eventos_con_datos(driver)
            
            # Guardar datos incrementalmente
            guardar_datos_incrementales(eventos_mes, archivo_csv)
            
            # Retroceder al mes anterior
            retroceder_mes(driver)
        
        except KeyboardInterrupt:
            print("Interrupción manual detectada. Terminando...")
            break

# Parámetros de iteración
fecha_limite = ""  # Ajustar según necesidad
archivo_csv = "eventos_incrementales.csv"

# Ejecutar el proceso
iterar_y_guardar(driver, fecha_limite, archivo_csv)

Se extrajeron 89 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 57 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 53 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 67 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Interrupción manual detectada. Terminando...


Este codigo tmb funciona tal como espero con la extracción de mes

In [49]:
def extraer_eventos_con_datos(driver):
    try:
        # Obtener el mes y año actual de la página
        try:
            mes_y_ano = driver.find_element(By.CLASS_NAME, "UyW9db").text.strip()
            anio = mes_y_ano[-4:] if mes_y_ano[-4:].isdigit() else "N.A."
        except Exception:
            mes_y_ano = "Sin información"  # Si no se encuentra, asignar valor por defecto
            anio = "N.A."
        
        # Encontrar todos los bloques de eventos (divs con class "qLWd9c")
        bloques_eventos = driver.find_elements(By.CLASS_NAME, "qLWd9c")
        
        datos = []
        
        # Iterar por cada bloque de evento
        for bloque in bloques_eventos:
            try:
                # 1. Extraer el número de eventos, día de la semana y fecha
                info_dia = bloque.find_element(By.CLASS_NAME, "CqwSk.XuJrye").text.strip()
                
                # **Extraer la última palabra del texto como "última palabra fecha"**
                try:
                    ultima_palabra_fecha = info_dia.split()[-1]  # Última palabra del texto
                except Exception:
                    ultima_palabra_fecha = "N.A."  # Valor predeterminado si falla la extracción
                
                # 2. Dentro del bloque, encontrar los eventos individuales
                eventos = bloque.find_elements(By.CLASS_NAME, "nHqeVd")
                
                for evento in eventos:
                    # a. Intentar extraer la hora (si no existe, usar "N.A.")
                    try:
                        hora = evento.find_element(By.CLASS_NAME, "DvyQhe.BdCDHc").text.strip()
                    except Exception:
                        hora = "N.A."  # Rellenar con "N.A." si no hay hora
                    
                    # b. Extraer la descripción y lugar
                    try:
                        descripcion_lugar = evento.find_element(By.CLASS_NAME, "WBi6vc").text.strip()
                    except Exception:
                        descripcion_lugar = "Sin información"  # Rellenar si no hay descripción
                    
                    # Almacenar los datos en un diccionario
                    datos.append({
                        "num_eventos,dia_semana,fecha": info_dia,
                        "ultima_palabra_fecha": ultima_palabra_fecha,  # Nueva columna
                        "hora": hora,
                        "descripción": descripcion_lugar,
                        "anio": anio,               # Año extraído del mes_y_ano
                        "mes_ano,ano": mes_y_ano
                    })
            except Exception as e:
                print(f"Error procesando un bloque de eventos: {e}")
        
        print(f"Se extrajeron {len(datos)} eventos.")
        return datos
    
    except Exception as e:
        print(f"Error al extraer eventos: {e}")
        return []

def guardar_datos_incrementales(datos, archivo_csv="eventos_incrementales.csv"):
    if datos:
        df = pd.DataFrame(datos)
        with open(archivo_csv, 'a', encoding='utf-8', newline='') as f:
            # Agregar encabezado solo si el archivo está vacío
            df.to_csv(f, index=False, header=f.tell() == 0)
        print(f"Datos guardados incrementalmente en {archivo_csv}.")
    else:
        print("No hay datos para guardar.")

def retroceder_mes(driver):
    try:
        boton_anterior = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//button[@aria-label="Mes anterior"]'))
        )
        boton_anterior.click()
        print("Retrocedimos al mes anterior.")
        # Esperar para permitir que la página cargue
        time.sleep(10)
    except Exception as e:
        print(f"Error al retroceder al mes anterior: {e}")

def iterar_y_guardar(driver, fecha_limite_texto, archivo_csv="eventos_incrementales.csv"):
    while True:
        try:
            # Obtener el mes y año actual
            try:
                mes_y_ano = driver.find_element(By.CLASS_NAME, "UyW9db").text.strip()
            except Exception:
                print("No se pudo obtener mes y año, terminando iteración.")
                break
            
            # Verificar si llegamos a la fecha límite
            if mes_y_ano <= fecha_limite_texto:
                print(f"Fecha límite alcanzada ({fecha_limite_texto}), terminando iteración.")
                break
            
            # Extraer eventos del mes actual
            eventos_mes = extraer_eventos_con_datos(driver)
            
            # Guardar datos incrementalmente
            guardar_datos_incrementales(eventos_mes, archivo_csv)
            
            # Retroceder al mes anterior
            retroceder_mes(driver)
        
        except KeyboardInterrupt:
            print("Interrupción manual detectada. Terminando...")
            break

# Parámetros de iteración
fecha_limite = ""
archivo_csv = "eventos_incrementales.csv"

# Ejecutar el proceso
iterar_y_guardar(driver, fecha_limite, archivo_csv)

Se extrajeron 55 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 89 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 57 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 53 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 67 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 52 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 49 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 81 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 94 eventos.
Datos guardados increm

In [54]:
def extraer_eventos_con_datos(driver):
    try:
        # Obtener el mes y año actual de la página
        try:
            mes_y_ano = driver.find_element(By.CLASS_NAME, "UyW9db").text.strip()
            anio = mes_y_ano[-4:] if mes_y_ano[-4:].isdigit() else "N.A."
        except Exception:
            mes_y_ano = "Sin información"  # Si no se encuentra, asignar valor por defecto
            anio = "N.A."
        
        # Encontrar todos los bloques de eventos (divs con class "qLWd9c")
        bloques_eventos = driver.find_elements(By.CLASS_NAME, "qLWd9c")
        
        datos = []
        
        # Iterar por cada bloque de evento
        for bloque in bloques_eventos:
            try:
                # 1. Extraer el número de eventos, día de la semana y fecha
                info_dia = bloque.find_element(By.CLASS_NAME, "CqwSk.XuJrye").text.strip()
                
                # Dividir el texto en 3 partes
                partes_info = info_dia.split(",")
                num_eventos = partes_info[0].strip() if len(partes_info) > 0 else "N.A."
                dia_semana = partes_info[1].strip() if len(partes_info) > 1 else "N.A."
                fecha = partes_info[2].strip() if len(partes_info) > 2 else "N.A."

                # Extraer la última palabra del texto como "última palabra fecha"
                try:
                    ultima_palabra_fecha = info_dia.split()[-1]  # Última palabra del texto
                except Exception:
                    ultima_palabra_fecha = "N.A."
                
                # 2. Dentro del bloque, encontrar los eventos individuales
                eventos = bloque.find_elements(By.CLASS_NAME, "nHqeVd")
                
                for evento in eventos:
                    # a. Intentar extraer la hora (si no existe, usar "N.A.")
                    try:
                        hora = evento.find_element(By.CLASS_NAME, "DvyQhe.BdCDHc").text.strip()
                    except Exception:
                        hora = "N.A."  # Rellenar con "N.A." si no hay hora
                    
                    # b. Extraer la descripción y lugar
                    try:
                        descripcion_lugar = evento.find_element(By.CLASS_NAME, "WBi6vc").text.strip()
                    except Exception:
                        descripcion_lugar = "Sin información"  # Rellenar si no hay descripción
                    
                    # Almacenar los datos en un diccionario
                    datos.append({
                        "num_eventos": num_eventos,
                        "dia_semana": dia_semana,
                        "fecha": fecha,
                        "ultima_palabra_fecha": ultima_palabra_fecha,
                        "hora": hora,
                        "descripcion": descripcion_lugar,
                        "anio": anio,
                        "mes_y_ano": mes_y_ano
                    })
            except Exception as e:
                print(f"Error procesando un bloque de eventos: {e}")
        
        print(f"Se extrajeron {len(datos)} eventos.")
        return datos
    
    except Exception as e:
        print(f"Error al extraer eventos: {e}")
        return []

def guardar_eventos_en_csv(datos, nombre_archivo="eventos_divididos.csv"):
    if datos:
        import pandas as pd
        df = pd.DataFrame(datos)
        df.to_csv(nombre_archivo, index=False)
        print(f"Datos guardados en {nombre_archivo}.")
    else:
        print("No hay datos para guardar.")

# Ejecución del proceso con archivo CSV
eventos_extraidos = extraer_eventos_con_datos(driver)
guardar_eventos_en_csv(eventos_extraidos, "eventos_divididos.csv")

Se extrajeron 35 eventos.
Datos guardados en eventos_divididos.csv.


In [58]:
def extraer_eventos_con_datos(driver):
    try:
        # Obtener el mes y año actual de la página
        try:
            mes_y_ano = driver.find_element(By.CLASS_NAME, "UyW9db").text.strip()
            anio = mes_y_ano[-4:] if mes_y_ano[-4:].isdigit() else "N.A."
        except Exception:
            mes_y_ano = "Sin información"  # Si no se encuentra, asignar valor por defecto
            anio = "N.A."
        
        # Encontrar todos los bloques de eventos (divs con class "qLWd9c")
        bloques_eventos = driver.find_elements(By.CLASS_NAME, "qLWd9c")
        
        datos = []
        
        # Iterar por cada bloque de evento
        for bloque in bloques_eventos:
            try:
                # 1. Extraer el número de eventos, día de la semana y fecha
                info_dia = bloque.find_element(By.CLASS_NAME, "CqwSk.XuJrye").text.strip()
                
                # Dividir el texto en 3 partes
                partes_info = info_dia.split(",")
                num_eventos = partes_info[0].strip() if len(partes_info) > 0 else "N.A."
                dia_semana = partes_info[1].strip() if len(partes_info) > 1 else "N.A."
                fecha = partes_info[2].strip() if len(partes_info) > 2 else "N.A."

                # Extraer la primera palabra de la fecha (que se espera sea un número)
                try:
                    primera_palabra_fecha = fecha.split()[0] if fecha else "N.A."
                except Exception:
                    primera_palabra_fecha = "N.A."
                
                # Extraer la última palabra del texto como "última palabra fecha"
                try:
                    ultima_palabra_fecha = info_dia.split()[-1]  # Última palabra del texto
                except Exception:
                    ultima_palabra_fecha = "N.A."
                
                # 2. Dentro del bloque, encontrar los eventos individuales
                eventos = bloque.find_elements(By.CLASS_NAME, "nHqeVd")
                
                for evento in eventos:
                    # a. Intentar extraer la hora (si no existe, usar "N.A.")
                    try:
                        hora = evento.find_element(By.CLASS_NAME, "DvyQhe.BdCDHc").text.strip()
                    except Exception:
                        hora = "N.A."  # Rellenar con "N.A." si no hay hora
                    
                    # b. Extraer la descripción y lugar
                    try:
                        descripcion_lugar = evento.find_element(By.CLASS_NAME, "WBi6vc").text.strip()
                    except Exception:
                        descripcion_lugar = "Sin información"  # Rellenar si no hay descripción
                    
                    # Almacenar los datos en un diccionario
                    datos.append({
                        "num_eventos": num_eventos,
                        "dia_semana": dia_semana,
                        "fecha": fecha,
                        "primera_palabra_fecha": primera_palabra_fecha,
                        "ultima_palabra_fecha": ultima_palabra_fecha,
                        "hora": hora,
                        "descripcion": descripcion_lugar,
                        "anio": anio,
                        "mes_y_ano": mes_y_ano
                    })
            except Exception as e:
                print(f"Error procesando un bloque de eventos: {e}")
        
        print(f"Se extrajeron {len(datos)} eventos.")
        return datos
    
    except Exception as e:
        print(f"Error al extraer eventos: {e}")
        return []

def guardar_eventos_en_csv(datos, nombre_archivo="eventos_divididos.csv"):
    if datos:
        import pandas as pd
        df = pd.DataFrame(datos)
        df.to_csv(nombre_archivo, index=False)
        print(f"Datos guardados en {nombre_archivo}.")
    else:
        print("No hay datos para guardar.")

# Ejecución del proceso con archivo CSV
eventos_extraidos = extraer_eventos_con_datos(driver)
guardar_eventos_en_csv(eventos_extraidos, "eventos_divididos.csv")

Se extrajeron 35 eventos.
Datos guardados en eventos_divididos.csv.


In [66]:
def extraer_eventos_con_datos(driver):
    try:
        # Obtener el mes y año actual de la página
        try:
            mes_y_ano = driver.find_element(By.CLASS_NAME, "UyW9db").text.strip()
            anio = mes_y_ano[-4:] if mes_y_ano[-4:].isdigit() else "N.A."
        except Exception:
            mes_y_ano = "Sin información"  # Si no se encuentra, asignar valor por defecto
            anio = "N.A."
        
        # Encontrar todos los bloques de eventos (divs con class "qLWd9c")
        bloques_eventos = driver.find_elements(By.CLASS_NAME, "qLWd9c")
        
        datos = []
        
        # Iterar por cada bloque de evento
        for bloque in bloques_eventos:
            try:
                # 1. Extraer el número de eventos, día de la semana y fecha
                info_dia = bloque.find_element(By.CLASS_NAME, "CqwSk.XuJrye").text.strip()
                
                # Dividir el texto en 3 partes
                partes_info = info_dia.split(",")
                num_eventos = partes_info[0].strip() if len(partes_info) > 0 else "N.A."
                dia_semana = partes_info[1].strip() if len(partes_info) > 1 else "N.A."
                fecha = partes_info[2].strip() if len(partes_info) > 2 else "N.A."

                # Extraer la primera palabra de la fecha (que se espera sea un número)
                try:
                    primera_palabra_fecha = fecha.split()[0] if fecha else "N.A."
                except Exception:
                    primera_palabra_fecha = "N.A."
                
                # Extraer la última palabra del texto como "última palabra fecha"
                try:
                    ultima_palabra_fecha = info_dia.split()[-1]  # Última palabra del texto
                except Exception:
                    ultima_palabra_fecha = "N.A."
                
                # 2. Dentro del bloque, encontrar los eventos individuales
                eventos = bloque.find_elements(By.CLASS_NAME, "nHqeVd")
                
                for evento in eventos:
                    # a. Intentar extraer la hora (si no existe, usar "N.A.")
                    try:
                        hora = evento.find_element(By.CLASS_NAME, "DvyQhe.BdCDHc").text.strip()
                    except Exception:
                        hora = "N.A."  # Rellenar con "N.A." si no hay hora
                    
                    # b. Extraer la descripción
                    try:
                        descripcion_lugar = evento.find_element(By.CLASS_NAME, "WBi6vc").text.strip()
                    except Exception:
                        descripcion_lugar = "Sin información"  # Rellenar si no hay descripción
                    
                    # Almacenar los datos en un diccionario
                    datos.append({
                        "num_eventos": num_eventos,
                        "dia_semana": dia_semana,
                        "fecha": fecha,
                        "primera_palabra_fecha": primera_palabra_fecha,
                        "ultima_palabra_fecha": ultima_palabra_fecha,
                        "hora": hora,
                        "anio": anio,
                        "mes_y_ano": mes_y_ano,
                        "descripcion": descripcion_lugar
                    })
            except Exception as e:
                print(f"Error procesando un bloque de eventos: {e}")
        
        print(f"Se extrajeron {len(datos)} eventos.")
        return datos
    
    except Exception as e:
        print(f"Error al extraer eventos: {e}")
        return []

def guardar_datos_incrementales(datos, archivo_csv="eventos_incrementales.csv"):
    if datos:
        df = pd.DataFrame(datos)
        with open(archivo_csv, 'a', encoding='utf-8', newline='') as f:
            # Agregar encabezado solo si el archivo está vacío
            df.to_csv(f, index=False, header=f.tell() == 0)
        print(f"Datos guardados incrementalmente en {archivo_csv}.")
    else:
        print("No hay datos para guardar.")

def retroceder_mes(driver):
    try:
        boton_anterior = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, '//button[@aria-label="Mes anterior"]'))
        )
        boton_anterior.click()
        print("Retrocedimos al mes anterior.")
        # Esperar para permitir que la página cargue
        time.sleep(10)
    except Exception as e:
        print(f"Error al retroceder al mes anterior: {e}")

def iterar_y_guardar(driver, fecha_limite_texto, archivo_csv="eventos_incrementales.csv"):
    while True:
        try:
            # Obtener el mes y año actual
            try:
                mes_y_ano = driver.find_element(By.CLASS_NAME, "UyW9db").text.strip()
            except Exception:
                print("No se pudo obtener mes y año, terminando iteración.")
                break
            
            # Verificar si llegamos a la fecha límite
            if mes_y_ano <= fecha_limite_texto:
                print(f"Fecha límite alcanzada ({fecha_limite_texto}), terminando iteración.")
                break
            
            # Extraer eventos del mes actual
            eventos_mes = extraer_eventos_con_datos(driver)
            
            # Guardar datos incrementalmente
            guardar_datos_incrementales(eventos_mes, archivo_csv)
            
            # Retroceder al mes anterior
            retroceder_mes(driver)
        
        except KeyboardInterrupt:
            print("Interrupción manual detectada. Terminando...")
            break

# Parámetros de iteración
fecha_limite = " "  # Ajustar según necesidad
archivo_csv = "eventos_incrementales.csv"

# Ejecutar el proceso
iterar_y_guardar(driver, fecha_limite, archivo_csv)

Se extrajeron 55 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 89 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 57 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 53 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 67 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 52 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 49 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 81 eventos.
Datos guardados incrementalmente en eventos_incrementales.csv.
Retrocedimos al mes anterior.
Se extrajeron 94 eventos.
Datos guardados increm